# Cumulative detections and period diffs

Two workflows over Amazon ACA (+ Andes supplemental) detections:

1. **Cumulatives** — union detections through each year / quarter tag and write dissolved GeoJSONs.
2. **Diffs** — incremental growth of each cumulative vs the previous tag; drop artefacts ≤ 11 ha; write `*-diff.geojson` under `diffs/`.

Timeline:
- year-ends 2018→2025
- 2025 quarters Q1–Q4 build on **through-2024** (not full-year 2025)
- 2026 quarters build on **full-year 2025** (not Q425)


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# --- parameters ---
t_main, t_iso, t_andes = 0.55, 0.8, 0.2
k, D = 5, 3
min_area_ha = 11.0
dissolve_buffer = 0.00001  # degrees, for cumulative dissolve
diff_buffer = 0.0001       # degrees, cleans 1-d artefacts after difference

basepath = Path("../data/outputs/48px_v4.10b-18d-20g-21a-22bc-ensemble")
post_dir = basepath / f"postprocessed_t{t_main}_d{k}_{D}km_t-iso{t_iso}"
andes_dir = basepath / "raw_detections"
cumul_dir = basepath / f"cumulative_t{t_main}_d{k}_{D}km_t-iso{t_iso}"
diff_dir = cumul_dir / "diffs"

cumul_dir.mkdir(parents=True, exist_ok=True)
diff_dir.mkdir(parents=True, exist_ok=True)

cumul_fname_stem = (
    f"Amazon_ACA_48px_v4.10b-18d-20g-21a-22bc-ensemble"
    f"_t{t_main}_d{k}_{D}km_t-iso{t_iso}_cumulative2018"
)


def amazon_path(period: str) -> Path:
    return (
        post_dir
        / (
            f"Amazon_ACA_48px_v4.10b-18d-20g-21a-22bc-ensemble_0.40_{period}"
            f"_t{t_main}_d{k}_{D}km_t-iso{t_iso}.geojson"
        )
    )


def andes_path(period: str) -> Path:
    return (
        andes_dir
        / (
            f"andes_supplemental_48px_v4.10b-18d-20g-21a-22bc-ensemble"
            f"_{t_andes}_{period}.geojson"
        )
    )


def cumul_path(tag: str) -> Path:
    return cumul_dir / f"{cumul_fname_stem}-{tag}.geojson"


def diff_outpath(tag: str) -> Path:
    return diff_dir / f"{cumul_fname_stem}-{tag}-diff.geojson"


def dissolve(df: gpd.GeoDataFrame, buffer_width: float = dissolve_buffer) -> gpd.GeoDataFrame:
    crs = df.crs or "EPSG:4326"
    dissolved = df.geometry.buffer(buffer_width, join_style=2).union_all()
    out = gpd.GeoDataFrame(geometry=[dissolved], crs=crs).explode(index_parts=False).reset_index(drop=True)
    out.geometry = out.geometry.buffer(-buffer_width, join_style=2)
    out = out.loc[out.geometry.notnull() & ~out.geometry.is_empty].reset_index(drop=True)
    return out


def write_cumulative(parts: list[Path], tag: str) -> Path:
    missing = [p for p in parts if not p.is_file()]
    if missing:
        raise FileNotFoundError("Missing inputs:\n" + "\n".join(str(p) for p in missing))

    frames = [gpd.read_file(p) for p in parts]
    for gdf in frames:
        if gdf.crs is None:
            gdf.set_crs("EPSG:4326", inplace=True)
    combined = gpd.GeoDataFrame(
        pd.concat(frames, ignore_index=True),
        crs=frames[0].crs,
    )
    cumul = dissolve(combined)
    out = cumul_path(tag)
    cumul.to_file(out, driver="GeoJSON", index=False)
    print(f"Wrote {out.name} from {len(parts)} inputs ({len(cumul)} polygons)")
    return out


def load_cumul(tag: str | None) -> gpd.GeoDataFrame:
    if tag is None:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    path = cumul_path(tag)
    if not path.is_file():
        raise FileNotFoundError(f"Missing cumulative file: {path.resolve()}")
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf.loc[gdf.geometry.notnull() & ~gdf.geometry.is_empty, ["geometry"]].copy()


def incremental_diff(
    current: gpd.GeoDataFrame,
    baseline: gpd.GeoDataFrame,
    *,
    buffer_width: float = diff_buffer,
    min_area_ha: float = min_area_ha,
) -> gpd.GeoDataFrame:
    """Geometry in current but not baseline; buffer-clean and drop small polygons."""
    if len(current) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=current.crs or "EPSG:4326")

    if baseline.crs is None:
        baseline = baseline.set_crs(current.crs)
    elif current.crs != baseline.crs:
        baseline = baseline.to_crs(current.crs)

    if len(baseline) == 0:
        diff = current.copy()
    else:
        diff = gpd.overlay(current, baseline, how="difference")

    diff = diff.loc[~diff.geometry.is_empty & diff.geometry.notnull()].copy()
    if len(diff) == 0:
        return gpd.GeoDataFrame(geometry=[], crs=current.crs)

    cleaned = diff.buffer(-buffer_width, join_style=2).buffer(buffer_width, join_style=2)
    diff = gpd.GeoDataFrame(geometry=cleaned, crs=current.crs)
    diff = diff.explode(index_parts=False).reset_index(drop=True)
    diff = diff.loc[~(diff.geometry.isnull() | diff.geometry.is_empty)].copy()
    if len(diff) == 0:
        return diff

    area_ha = diff.to_crs("EPSG:3857").geometry.area / 1e4
    return diff.loc[area_ha > min_area_ha].reset_index(drop=True)


print(f"basepath:   {basepath.resolve()}")
print(f"cumul_dir:  {cumul_dir.resolve()}")
print(f"diff_dir:   {diff_dir.resolve()}")


## 1. Compute cumulatives

Concatenates Amazon postprocessed detections + Andes supplemental for each tag, dissolves, and writes to `cumul_dir`.


In [ ]:
years = list(range(2018, 2026))  # 2018..2025
year_periods = [f"{y}-01-01_{y}-12-31" for y in years]
amazon_years = [amazon_path(p) for p in year_periods]
andes_years = [andes_path(p) for p in year_periods]

# Year-end cumulatives
for i, y in enumerate(years):
    write_cumulative(amazon_years[: i + 1] + andes_years[: i + 1], str(y))

# 2025 quarters: through-2024 only + progressive 2025 quarters (exclude full-year 2025)
quarters2025 = [
    "2025-01-01_2025-03-31",
    "2025-04-01_2025-06-30",
    "2025-07-01_2025-09-30",
    "2025-10-01_2025-12-31",
]
amazon_q25 = [amazon_path(p) for p in quarters2025]
andes_q25 = [andes_path(p) for p in quarters2025]
base_through_2024 = amazon_years[:-1] + andes_years[:-1]

for i, q in enumerate(range(1, 5)):
    write_cumulative(
        base_through_2024 + amazon_q25[: i + 1] + andes_q25[: i + 1],
        f"Q{q}25",
    )

# 2026 quarters: full years through 2025 (no 2025 quarters) + progressive 2026 quarters
quarters2026 = ["2026-01-01_2026-03-31", "2026-04-01_2026-06-30"]
amazon_q26 = [amazon_path(p) for p in quarters2026]
andes_q26 = [andes_path(p) for p in quarters2026]
base_through_2025 = amazon_years + andes_years

for i, q in enumerate(range(1, 3)):
    write_cumulative(
        base_through_2025 + amazon_q26[: i + 1] + andes_q26[: i + 1],
        f"Q{q}26",
    )


## 2. Compute diffs

For each tag, write incremental growth vs the previous cumulative to `diff_dir` as `<fname>-diff.geojson`.

| Current | Baseline |
|---|---|
| 2018 | empty |
| 2019…2025 | prior year |
| Q125 | 2024 |
| Q225…Q425 | prior quarter |
| Q126 | full-year 2025 |
| Q226 | Q126 |


In [ ]:
diff_pairs = (
    [("2018", None)]
    + [(str(y), str(y - 1)) for y in range(2019, 2026)]
    + [("Q125", "2024")]
    + [(f"Q{q}25", f"Q{q - 1}25") for q in range(2, 5)]
    + [("Q126", "2025")]
    + [("Q226", "Q126")]
)

for tag, prev in diff_pairs:
    growth = incremental_diff(load_cumul(tag), load_cumul(prev))
    out = diff_outpath(tag)
    growth.to_file(out, driver="GeoJSON", index=False)

    n_poly = len(growth)
    area_ha = (
        float(growth.to_crs("EPSG:3857").geometry.area.sum() / 1e4) if n_poly else 0.0
    )
    print(
        f"{tag} − {prev or '∅'}: {n_poly} polygons, {area_ha:.1f} ha → {out}"
    )
